In [1]:
%cd /home/brimmann/works/xRAG

/home/brimmann/works/xRAG


/home/brimmann/works/xRAG/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
import torch
import torch.nn.functional as F
from torch.optim import Adam
from transformers import AutoTokenizer

# Import your custom model and config classes
from src.model.xMistral.modeling_xmistral import XMistralForCausalLM, XMistralConfig
from src.distill.modeling_xgemma import XGemmaForCausalLM, XGemmaConfig

# --- Configuration for the Validation Run ---
teacher_base_model_name = "mistralai/Mistral-7B-v0.1"
student_base_model_name = "google/gemma-2-2b"
retriever_hidden_size = 128  # An arbitrary dimension for this test
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- Validation Setup ---")
print(f"Device: {device}")
print(f"Mock Teacher Base: {teacher_base_model_name}")
print(f"Student Base: {student_base_model_name}")

--- Validation Setup ---
Device: cuda
Mock Teacher Base: mistralai/Mistral-7B-v0.1
Student Base: google/gemma-2-2b


In [3]:
student_config = XGemmaConfig.from_pretrained(
    student_base_model_name,
    retriever_hidden_size=retriever_hidden_size,
    projector_type='mlp2x_gelu'
)
student_model = XGemmaForCausalLM.from_pretrained(
    student_base_model_name,
    config=student_config,
    device_map="auto",
    offload_folder="offload_student",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some weights of XGemmaForCausalLM were not initialized from the model checkpoint at google/gemma-2-2b and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
student_model

XGemmaForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNo

In [ ]:
# We will use the student's tokenizer for both models in this test
from accelerate.utils import offload


tokenizer = AutoTokenizer.from_pretrained(student_base_model_name)
xrag_token = "<xRAG>"
tokenizer.add_special_tokens({"additional_special_tokens": [xrag_token]})
xrag_token_id = tokenizer.convert_tokens_to_ids(xrag_token)

# --- Configure Teacher ---
print("\nConfiguring mock teacher...")
teacher_config = XMistralConfig.from_pretrained(
    teacher_base_model_name,
    retriever_hidden_size=retriever_hidden_size,
    projector_type='mlp2x_gelu',
)
teacher_model = XMistralForCausalLM.from_pretrained(teacher_base_model_name,
config=teacher_config,
device_map="auto",
offload_folder="offload_teacher",
torch_dtype=torch.bfloat16,
low_cpu_mem_usage=True
)
teacher_model.resize_token_embeddings(len(tokenizer)) # Adjust for Gemma's tokenizer
teacher_model.set_xrag_token_id(xrag_token_id)
# teacher_model.to(device)
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False
print("Mock teacher configured and frozen.")

# --- Configure Student ---
print("\nConfiguring student...")
student_config = XGemmaConfig.from_pretrained(
    student_base_model_name,
    retriever_hidden_size=retriever_hidden_size,
    projector_type='mlp2x_gelu'
)
student_model = XGemmaForCausalLM.from_pretrained(
    student_base_model_name,
    config=student_config,
    device_map="auto",
    offload_folder="offload_student",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True)
student_model.resize_token_embeddings(len(tokenizer))
student_model.set_xrag_token_id(xrag_token_id)
# student_model.to(device)
student_model.train()
for name, param in student_model.named_parameters():
    if "projector" not in name:
        param.requires_grad = False
print("Student configured. Base model frozen.")

# --- Verify Trainable Parameters ---
print("\nTrainable parameters in student model:")
trainable_params = [name for name, param in student_model.named_parameters() if param.requires_grad]
if trainable_params:
    for name in trainable_params:
        print(name)
else:
    print("Warning: No trainable parameters found in student model!")


Configuring mock teacher...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-v0.1 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some parameters are on the meta device because they were offloaded to the cpu.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
# The optimizer will only act on the trainable parameters (the projector)
optimizer = Adam(student_model.projector.parameters(), lr=0.1)
distillation_loss_fn = torch.nn.KLDivLoss(reduction='batchmean')
temperature = 2.0

print("Optimizer and loss function are set up.")

In [ ]:
print("\n--- Performing a single validation training step ---")

# 1. Create a dummy batch of data
prompt = f"This is a test prompt with a special token: {xrag_token}."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
num_xrag_tokens = torch.sum(input_ids == xrag_token_id).item()
dummy_retrieval_embeds = torch.randn(num_xrag_tokens, retriever_hidden_size).to(device)

# 2. Store a projector weight from the student *before* the update
# FIX: Access the inner .projector which is the nn.Sequential
first_projector_weight_before = student_model.projector.projector[0].weight.data[0, 0].clone()

# 3. Get teacher's output (no gradients needed)
with torch.no_grad():
    teacher_outputs = teacher_model(input_ids=input_ids, retrieval_embeds=dummy_retrieval_embeds)
    teacher_logits = teacher_outputs.logits

# 4. Get student's output (gradients will be computed here)
student_outputs = student_model(input_ids=input_ids, retrieval_embeds=dummy_retrieval_embeds)
student_logits = student_outputs.logits

# 5. Calculate loss
# Align sequence lengths if teacher and student models produce different length outputs
min_seq_len = min(teacher_logits.shape[1], student_logits.shape[1])
loss = distillation_loss_fn(
    F.log_softmax(student_logits[:, :min_seq_len, :] / temperature, dim=-1),
    F.softmax(teacher_logits[:, :min_seq_len, :] / temperature, dim=-1)
)

# 6. Backpropagate and update weights
optimizer.zero_grad()
loss.backward()

# --- Verification ---
# FIX: Access the inner .projector to check the gradient
grad_exists = student_model.projector.projector[0].weight.grad is not None
print(f"Gradient was computed for the projector: {grad_exists}")

optimizer.step()

# 7. Get the same projector weight *after* the update
# FIX: Access the inner .projector again
first_projector_weight_after = student_model.projector.projector[0].weight.data[0, 0].clone()

print(f"\nProjector weight before update: {first_projector_weight_before.item()}")
print(f"Projector weight after update:  {first_projector_weight_after.item()}")

# 8. Final Check
if first_projector_weight_before != first_projector_weight_after and grad_exists:
    print("\n✅ SUCCESS: The validation run was successful. Projector weights were updated.")
else:
    print("\n❌ FAILURE: The validation run failed. Projector weights were NOT updated.")